In [1]:
// 1. Imports Ivy (Utilise le double :: pour l'auto-détection de version)
import $ivy.`org.apache.spark::spark-sql:3.5.3`
import $ivy.`org.apache.spark::spark-graphx:3.5.3`


import $ivy.$
import $ivy.$

In [2]:
import org.apache.spark.SparkConf
import org.apache.spark.sql.SparkSession
import org.apache.spark.graphx._

// 1. Définition de la configuration
val conf = new SparkConf()
  .setAppName("SparkApp-Scala")
  .setMaster("spark://spark:7077")
  // On ajoute GraphX à tes packages existants
  .set("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.spark:spark-hadoop-cloud_2.12:3.5.3,org.apache.spark:spark-graphx_2.12:3.5.3")
  // Committer S3A
  .set("spark.hadoop.fs.s3a.committer.name", "staging")
  .set("spark.hadoop.mapreduce.outputcommitter.factory.scheme.s3a", "org.apache.hadoop.fs.s3a.commit.S3ACommitterFactory")
  .set("spark.hadoop.fs.s3a.committer.staging.tmp.path", "/tmp/s3a-commit")
  .set("spark.hadoop.fs.s3a.committer.staging.unique-filenames", "true")
  .set("spark.hadoop.fs.s3a.committer.staging.conflict-mode", "replace")

// 2. Initialisation de la Session
val spark = SparkSession.builder().config(conf).getOrCreate()
val sc = spark.sparkContext

// 3. Configuration S3 (Garage)
val keyId = "GKd8a9e890f723a329bf5f6822"
val secretKey = "7b77d728326c542d8b9cd61f5d94c3f3a1e193548ef0329b3a4c7f21c0974524"
val garageIp = "garage"

val hadoopConf = sc.hadoopConfiguration
hadoopConf.set("fs.s3a.endpoint", s"http://$garageIp:3900")
hadoopConf.set("fs.s3a.access.key", keyId)
hadoopConf.set("fs.s3a.secret.key", secretKey)
hadoopConf.set("fs.s3a.endpoint.region", "garage")
hadoopConf.set("fs.s3a.path.style.access", "true")
hadoopConf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoopConf.set("fs.s3a.connection.ssl.enabled", "false")

println("✅ Environnement Scala/Spark/S3 prêt !")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/10 14:22:06 INFO SparkContext: Running Spark version 3.5.3
26/03/10 14:22:06 INFO SparkContext: OS info Linux, 6.6.87.2-microsoft-standard-WSL2, amd64
26/03/10 14:22:06 INFO SparkContext: Java version 1.8.0_442
26/03/10 14:22:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/10 14:22:06 INFO ResourceUtils: ==============================================================
26/03/10 14:22:06 INFO ResourceUtils: No custom resources configured for spark.driver.
26/03/10 14:22:06 INFO ResourceUtils: ==============================================================
26/03/10 14:22:06 INFO SparkContext: Submitted application: SparkApp-Scala
26/03/10 14:22:06 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, scr

✅ Environnement Scala/Spark/S3 prêt !


import org.apache.spark.SparkConf
import org.apache.spark.sql.SparkSession
import org.apache.spark.graphx._
conf: SparkConf = org.apache.spark.SparkConf@39f8c23c
spark: SparkSession = org.apache.spark.sql.SparkSession@4cb976fd
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@bea9e3
keyId: String = "GKd8a9e890f723a329bf5f6822"
secretKey: String = "7b77d728326c542d8b9cd61f5d94c3f3a1e193548ef0329b3a4c7f21c0974524"
garageIp: String = "garage"
hadoopConf: org.apache.hadoop.conf.Configuration = Configuration: core-default.xml, core-site.xml, mapred-default.xml, mapred-site.xml, yarn-default.xml, yarn-site.xml, __spark_hadoop_conf__.xml

In [3]:
import org.apache.spark.rdd.RDD

// Création de 3 sommets : Alice, Bob et Charlie
val users: RDD[(VertexId, String)] = sc.parallelize(Array(
  (1L, "Alice"), (2L, "Bob"), (3L, "Charlie")
))

// Création de 3 arêtes pour former un triangle
val relationships: RDD[Edge[String]] = sc.parallelize(Array(
  Edge(1L, 2L, "suit"), 
  Edge(2L, 3L, "suit"), 
  Edge(3L, 1L, "suit")
))

// Construction du graphe
val graph = Graph(users, relationships)

// --- LES TESTS ---

// Test 1 : Compter les sommets
val vCount = graph.vertices.count()

// Test 2 : Filtrer les relations
val eCount = graph.edges.filter(e => e.attr == "suit").count()

println(s"\n✅ TEST RÉUSSI !")
println(s"Nombre de sommets détectés : $vCount")
println(s"Nombre d'arêtes détectées : $eCount")

// Test 3 : Affichage des triplets (La base de GraphX)

graph.triplets.collect().foreach(t => 
  println(s"${t.srcAttr} ---> [${t.attr}] ---> ${t.dstAttr}")
)

26/03/10 14:22:13 INFO SparkContext: Starting job: fold at VertexRDDImpl.scala:90
26/03/10 14:22:13 INFO DAGScheduler: Registering RDD 6 (mapPartitions at VertexRDD.scala:356) as input to shuffle 1
26/03/10 14:22:13 INFO DAGScheduler: Registering RDD 0 (parallelize at cmd3.sc:4) as input to shuffle 0
26/03/10 14:22:13 INFO DAGScheduler: Got job 0 (fold at VertexRDDImpl.scala:90) with 2 output partitions
26/03/10 14:22:13 INFO DAGScheduler: Final stage: ResultStage 2 (fold at VertexRDDImpl.scala:90)
26/03/10 14:22:13 INFO DAGScheduler: Parents of final stage: List(ShuffleMapStage 0, ShuffleMapStage 1)
26/03/10 14:22:13 INFO DAGScheduler: Missing parents: List(ShuffleMapStage 0, ShuffleMapStage 1)
26/03/10 14:22:13 INFO DAGScheduler: Submitting ShuffleMapStage 0 (VertexRDD.createRoutingTables - vid2pid (aggregation) MapPartitionsRDD[6] at mapPartitions at VertexRDD.scala:356), which has no missing parents
26/03/10 14:22:13 INFO MemoryStore: Block broadcast_0 stored as values in memory (e

org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 3.0 failed 4 times, most recent failure: Lost task 0.3 in stage 3.0 (TID 12) (172.24.0.3 executor 0): java.lang.ClassCastException: cannot assign instance of java.lang.invoke.SerializedLambda to field org.apache.spark.rdd.MapPartitionsRDD.f of type scala.Function3 in instance of org.apache.spark.rdd.MapPartitionsRDD
	at java.base/java.io.ObjectStreamClass$FieldReflector.setObjFieldValues(ObjectStreamClass.java:2096)
	at java.base/java.io.ObjectStreamClass$FieldReflector.checkObjectFieldValueTypes(ObjectStreamClass.java:2060)
	at java.base/java.io.ObjectStreamClass.checkObjFieldValueTypes(ObjectStreamClass.java:1347)
	at java.base/java.io.ObjectInputStream$FieldValues.defaultCheckFieldValues(ObjectInputStream.java:2679)
	at java.base/java.io.ObjectInputStream.readSerialData(ObjectInputStream.java:2486)
	at java.base/java.io.ObjectInputStream.readOrdinaryObject(ObjectInputStream.java:2257)
	at java.base/ja